# Augmented TDS Training
Trains the baseline TDS model with three additional data augmentations:
- **GaussianNoise** (std=0.05) — additive noise on raw EMG
- **RandomAmplitudeScale** (0.75–1.33×) — random gain on raw EMG
- **ChannelDropout** (p=0.1) — randomly zero entire electrode channels

Baseline (no augmentation) test/CER: **22.09**


In [ ]:
import os
# Set cudnn path for GPU Colab (T4/A100)
for cudnn_path in [
    "/usr/local/lib/python3.10/dist-packages/nvidia/cudnn/lib",
    "/usr/local/lib/python3.11/dist-packages/nvidia/cudnn/lib",
]:
    if os.path.exists(cudnn_path):
        os.environ["LD_LIBRARY_PATH"] = cudnn_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        print(f"Set LD_LIBRARY_PATH: {cudnn_path}")
        break


In [ ]:
# Augmented single-user training
# New transforms active by default via config/transforms/log_spectrogram.yaml
!python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 \
  batch_size=128 \
  num_workers=7


In [ ]:
# To resume from a checkpoint, uncomment and set the path:
# CKPT = "logs/<date>/<time>/checkpoints/last.ckpt"
# !python -m emg2qwerty.train \
#   user="single_user" \
#   trainer.accelerator=gpu trainer.devices=1 \
#   +trainer.precision=16 \
#   batch_size=128 \
#   num_workers=7 \
#   checkpoint=


In [ ]:
# Testing — set CKPT to best checkpoint path from training output
CKPT = "logs/<date>/<time>/checkpoints/<best>.ckpt"  # TODO: update this
!python -m emg2qwerty.train \
  user="single_user" \
  checkpoint= \
  train=False \
  trainer.accelerator=gpu \
  decoder=ctc_greedy
